<a href="https://colab.research.google.com/github/Fahad-Hafeez/phishing-ml-classifier-comparison/blob/main/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas statsmodels imbalanced-learn

import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.svm import SVC
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv('uci-ml-phishing-dataset.csv')

In [ ]:
print("Shape:", df.shape)
print(df.head())
print(df.info())
print(df.describe())

In [ ]:
print(df['Result'].value_counts())

In [ ]:
df['Result'].value_counts().plot(kind="bar", title='Class Distribution')
print('1 (Phishing) % =', (6157/11056)*100)
print('-1 (Legitimate) % =', (4898/11056)*100)

In [ ]:
class_counts = df['Result'].value_counts()

total_instances = class_counts.sum()
phishing_count = class_counts.get(1, 0)
legitimate_count = class_counts.get(-1, 0)

phishing_percent = (phishing_count / total_instances) * 100
legitimate_percent = (legitimate_count / total_instances) * 100

# Adjust figsize to full-width IEEE layout (7x5 inches)
plt.figure(figsize=(7, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')

plt.xticks(ticks=[0, 1], labels=['Phishing', 'Legitimate'])
plt.xlabel('Class')
plt.ylabel('Number of Instances')
plt.title('Class Distribution of Phishing Websites Dataset')

# Add text annotations on top of the bars
for index, value in enumerate(class_counts.values):
    plt.text(index, value + 50, str(value), ha='center', va='bottom')

plt.tight_layout()
# Save with dpi=300 and bbox_inches='tight' as requested
plt.savefig('fig1_class_distribution.pdf', dpi=300, bbox_inches='tight')
plt.show()

caption = f"Figure 1: Class distribution of the UCI Phishing Websites dataset (n = {total_instances}). Phishing instances: {phishing_count} ({phishing_percent:.2f}%). Legitimate instances: {legitimate_count} ({legitimate_percent:.2f}%)."
print(caption)

In [ ]:
print(df.isnull().sum())

In [ ]:
#Count the number of distinct elements
print(df.nunique())

In [ ]:
#Extract the unique values for every column
print(df.apply(lambda x: x.unique()))

In [ ]:
X = df.drop('Result', axis=1)
y = df['Result']

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

In [ ]:
SEED = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=SEED, stratify=y)

print(f"X Train shape: {X_train.shape}")
print(f"X Test shape: {X_test.shape}")
print(f"y Train shape: {y_train.shape}")
print(f"y Test shape: {y_test.shape}")

print("\nClass distribution in y_train:")
print(y_train.value_counts())

print("\nClass distribution in y_test:")
print(y_test.value_counts())

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")
print(f"Shape of scaled X_train: {X_train_scaled.shape}")
print(f"Shape of scaled X_test: {X_test_scaled.shape}")

In [ ]:
sm = SMOTE(random_state=SEED)
X_train_balanced, y_train_balanced = sm.fit_resample(X_train_scaled, y_train)

print(f"Shape of X_train after SMOTE: {X_train_balanced.shape}")
print(f"Shape of y_train after SMOTE: {y_train_balanced.shape}")
print("Class distribution in y_train after SMOTE:")
print(y_train_balanced.value_counts())

In [ ]:
param_grid = [
    {
        'penalty': ['l1'],
        'solver': ['liblinear', 'saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'penalty': ['l2'],
        'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'penalty': ['elasticnet'],
        'solver': ['saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'l1_ratio': [0.1, 0.5, 0.9] # Added l1_ratio for elasticnet
    }
]
GSCV = GridSearchCV(LogisticRegression(random_state=SEED, max_iter=1000), param_grid, cv=5, scoring='f1_macro', n_jobs=1)
GSCV.fit(X_train_scaled, y_train)
print(GSCV.best_params_, GSCV.best_score_)

In [ ]:
best_lr = GSCV.best_estimator_
print(f'Best model:', best_lr)

In [ ]:
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}
RandomForest = GridSearchCV(RandomForestClassifier(random_state=SEED), rf_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
RandomForest.fit(X_train_scaled, y_train)
print(RandomForest.best_params_, RandomForest.best_score_)

In [ ]:
best_rf = RandomForest.best_estimator_
print(best_rf)

In [ ]:
svm_param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}
SVM = GridSearchCV(SVC(random_state=SEED, probability=True), svm_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
SVM.fit(X_train_scaled, y_train)
print(SVM.best_params_, SVM.best_score_)

In [ ]:
best_svm = SVM.best_estimator_
print(best_svm)

In [ ]:
def evaluate_model(model, X_test, y_test):
    """
    Evaluates a trained model on test data and returns a dictionary of metrics.

    Args:
        model: A trained scikit-learn model with predict and predict_proba methods.
        X_test: Test features.
        y_test: True labels for the test data.

    Returns:
        A dictionary containing accuracy, precision, recall, f1-score, and auc-roc.
    """
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')

    auc_roc = None
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)
        # Determine the index of the positive class (1) in the model's classes_
        if 1 in model.classes_:
            pos_class_idx = list(model.classes_).index(1)
            auc_roc = roc_auc_score(y_test, y_proba[:, pos_class_idx], average='macro')
        else:
            print("Warning: Positive class (1) not found in model.classes_. AUC-ROC not calculated.")
    else:
        print("Warning: Model does not have predict_proba method. AUC-ROC score cannot be calculated.")

    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'auc_roc': auc_roc
    }
    return metrics

In [ ]:
lr_eval = evaluate_model(best_lr, X_test_scaled, y_test)
print(f"Logistic Regression Evaluation:", lr_eval)

In [ ]:
rf_eval = evaluate_model(best_rf, X_test_scaled, y_test)
print(f"Random Forest Evaluation:", rf_eval)

In [ ]:
svm_eval = evaluate_model(best_svm, X_test_scaled, y_test)
print(f"SVM Evaluation:", svm_eval)

In [ ]:
pred_lr = best_lr.predict(X_test_scaled)
pred_rf = best_rf.predict(X_test_scaled)
pred_svm = best_svm.predict(X_test_scaled)

results = [
    {'Model': 'Logistic Regression', **lr_eval},
    {'Model': 'Random Forest', **rf_eval},
    {'Model': 'SVM', **svm_eval}
]

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
param_grid = [
    {
        'penalty': ['l1'],
        'solver': ['liblinear', 'saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'penalty': ['l2'],
        'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'penalty': ['elasticnet'],
        'solver': ['saga'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'l1_ratio': [0.1, 0.5, 0.9] # Added l1_ratio for elasticnet
    }
]
GSCV_balanced = GridSearchCV(LogisticRegression(random_state=SEED, max_iter=1000), param_grid, cv=5, scoring='f1_macro', n_jobs=1)
GSCV_balanced.fit(X_train_balanced, y_train_balanced)
print(GSCV_balanced.best_params_, GSCV.best_score_)

In [ ]:
best_lr_bal = GSCV_balanced.best_estimator_
print(f'Best model:', best_lr_bal)

In [ ]:
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}
RandomForest_balanced = GridSearchCV(RandomForestClassifier(random_state=SEED), rf_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
RandomForest_balanced.fit(X_train_balanced, y_train_balanced)
print(RandomForest.best_params_, RandomForest.best_score_)

In [ ]:
best_rf_bal = RandomForest_balanced.best_estimator_
print(f'Best model:', best_rf_bal)

In [ ]:
svm_param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}
SVM_balanced = GridSearchCV(SVC(random_state=SEED, probability=True), svm_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
SVM_balanced.fit(X_train_balanced, y_train_balanced)
print(SVM.best_params_, SVM.best_score_)

In [ ]:
best_svm_bal = SVM_balanced.best_estimator_
print(f'Best model:', best_svm_bal)

In [ ]:
lr_eval_bal = evaluate_model(best_lr_bal, X_test_scaled, y_test)
rf_eval_bal = evaluate_model(best_rf_bal, X_test_scaled, y_test)
svm_eval_bal = evaluate_model(best_svm_bal, X_test_scaled, y_test)

results = [
    {'Model': 'Logistic Regression (Unbalanced)', **lr_eval},
    {'Model': 'Random Forest (Unbalanced)', **rf_eval},
    {'Model': 'SVM (Unbalanced)', **svm_eval},
    {'Model': 'Logistic Regression (Balanced)', **lr_eval_bal},
    {'Model': 'Random Forest (Balanced)', **rf_eval_bal},
    {'Model': 'SVM (Balanced)', **svm_eval_bal}
]

results_df = pd.DataFrame(results)
print(results_df)


## Expanded SVM Hyperparameter Tuning and Evaluation

In [ ]:
from sklearn.metrics import classification_report

# Define the expanded SVM parameter grid
expanded_svm_param_grid = [
    {
        'C': [0.01, 0.1, 1, 10, 100, 1000],
        'kernel': ['linear'],
        'gamma': ['scale'] # gamma is ignored for linear kernel, 'scale' is a dummy value
    },
    {
        'C': [0.01, 0.1, 1, 10, 100, 1000],
        'kernel': ['rbf'],
        'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1.0]
    }
]

# --- Expanded SVM Tuning (Unbalanced Data) ---
print("\n--- Expanded SVM Tuning (Unbalanced Data) ---")
SVM_expanded = GridSearchCV(
    SVC(random_state=SEED, probability=True),
    expanded_svm_param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
SVM_expanded.fit(X_train_scaled, y_train)
best_svm_expanded = SVM_expanded.best_estimator_

print("\nBest parameters for Unbalanced SVM:", SVM_expanded.best_params_)
print("Best F1-macro score for Unbalanced SVM:", SVM_expanded.best_score_)

# Check C boundary for unbalanced SVM
optimal_C_unbalanced = SVM_expanded.best_params_['C']
min_C = min(p['C'] for p in expanded_svm_param_grid for C_val in p['C'])
max_C = max(p['C'] for p in expanded_svm_param_grid for C_val in p['C'])

if optimal_C_unbalanced == min(expanded_svm_param_grid[0]['C']) or optimal_C_unbalanced == max(expanded_svm_param_grid[0]['C']):
    print(f"\nWARNING: Optimal C ({optimal_C_unbalanced}) for Unbalanced SVM is at the boundary of the search range. Consider expanding the C grid further.")
else:
    print(f"\nOptimal C ({optimal_C_unbalanced}) for Unbalanced SVM is in the interior of the search range.")

# --- Evaluation of Expanded Unbalanced SVM on Test Set ---
print("\n--- Evaluation of Expanded Unbalanced SVM on Test Set ---")
svm_eval_expanded = evaluate_model(best_svm_expanded, X_test_scaled, y_test)
print("Expanded Unbalanced SVM Evaluation:", svm_eval_expanded)

pred_svm_expanded = best_svm_expanded.predict(X_test_scaled)
print("\nClassification Report (Expanded Unbalanced SVM):")
print(classification_report(y_test, pred_svm_expanded, target_names=['Legitimate', 'Phishing']))

# Confusion Matrix for Expanded Unbalanced SVM
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, pred_svm_expanded)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Legitimate', 'Phishing'], yticklabels=['Legitimate', 'Phishing'])
ax.set_title('Confusion Matrix: Expanded Unbalanced SVM')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix_expanded_unbalanced_svm.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Expanded SVM Tuning (Balanced Data) ---
print("\n--- Expanded SVM Tuning (Balanced Data) ---")
SVM_balanced_expanded = GridSearchCV(
    SVC(random_state=SEED, probability=True),
    expanded_svm_param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
SVM_balanced_expanded.fit(X_train_balanced, y_train_balanced)
best_svm_bal_expanded = SVM_balanced_expanded.best_estimator_

print("\nBest parameters for Balanced SVM:", SVM_balanced_expanded.best_params_)
print("Best F1-macro score for Balanced SVM:", SVM_balanced_expanded.best_score_)

# Check C boundary for balanced SVM
optimal_C_balanced = SVM_balanced_expanded.best_params_['C']
min_C = min(p['C'] for p in expanded_svm_param_grid for C_val in p['C'])
max_C = max(p['C'] for p in expanded_svm_param_grid for C_val in p['C'])

if optimal_C_balanced == min(expanded_svm_param_grid[0]['C']) or optimal_C_balanced == max(expanded_svm_param_grid[0]['C']):
    print(f"\nWARNING: Optimal C ({optimal_C_balanced}) for Balanced SVM is at the boundary of the search range. Consider expanding the C grid further.")
else:
    print(f"\nOptimal C ({optimal_C_balanced}) for Balanced SVM is in the interior of the search range.")

# --- Evaluation of Expanded Balanced SVM on Test Set ---
print("\n--- Evaluation of Expanded Balanced SVM on Test Set ---")
svm_eval_bal_expanded = evaluate_model(best_svm_bal_expanded, X_test_scaled, y_test)
print("Expanded Balanced SVM Evaluation:", svm_eval_bal_expanded)

pred_svm_bal_expanded = best_svm_bal_expanded.predict(X_test_scaled)
print("\nClassification Report (Expanded Balanced SVM):")
print(classification_report(y_test, pred_svm_bal_expanded, target_names=['Legitimate', 'Phishing']))

# Confusion Matrix for Expanded Balanced SVM
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, pred_svm_bal_expanded)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Legitimate', 'Phishing'], yticklabels=['Legitimate', 'Phishing'])
ax.set_title('Confusion Matrix: Expanded Balanced SVM')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix_expanded_balanced_svm.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Summary Comparison Table ---
print("\n--- Summary Comparison Table ---")

# Ensure original SVM evaluation results are available
# If these variables are not globally defined from previous executions,
# they would need to be re-computed or loaded.
# Assuming svm_eval and svm_eval_bal are from cells 6NTCiN8vsNqW and FahePKYF489n

summary_data = [
    {
        'Model': 'Original SVM (Unbalanced)',
        'Optimal C': SVM.best_params_['C'],
        'Optimal Kernel': SVM.best_params_['kernel'],
        'Optimal Gamma': SVM.best_params_['gamma'],
        'Test F1-macro': svm_eval['f1_score'],
        'Test Accuracy': svm_eval['accuracy'],
        'Test AUC-ROC': svm_eval['auc_roc']
    },
    {
        'Model': 'Expanded SVM (Unbalanced)',
        'Optimal C': best_svm_expanded.best_params_['C'],
        'Optimal Kernel': best_svm_expanded.best_params_['kernel'],
        'Optimal Gamma': best_svm_expanded.best_params_['gamma'],
        'Test F1-macro': svm_eval_expanded['f1_score'],
        'Test Accuracy': svm_eval_expanded['accuracy'],
        'Test AUC-ROC': svm_eval_expanded['auc_roc']
    },
    {
        'Model': 'Original SVM (Balanced)',
        'Optimal C': SVM_balanced.best_params_['C'],
        'Optimal Kernel': SVM_balanced.best_params_['kernel'],
        'Optimal Gamma': SVM_balanced.best_params_['gamma'],
        'Test F1-macro': svm_eval_bal['f1_score'],
        'Test Accuracy': svm_eval_bal['accuracy'],
        'Test AUC-ROC': svm_eval_bal['auc_roc']
    },
    {
        'Model': 'Expanded SVM (Balanced)',
        'Optimal C': best_svm_bal_expanded.best_params_['C'],
        'Optimal Kernel': best_svm_bal_expanded.best_params_['kernel'],
        'Optimal Gamma': best_svm_bal_expanded.best_params_['gamma'],
        'Test F1-macro': svm_eval_bal_expanded['f1_score'],
        'Test Accuracy': svm_eval_bal_expanded['accuracy'],
        'Test AUC-ROC': svm_eval_bal_expanded['auc_roc']
    }
]

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

In [ ]:
!pip install xgboost
from xgboost import XGBClassifier

# Remap labels: XGBoost requires 0/1 labels (not -1/1)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_xgb     = le.fit_transform(y_train)           # -1→0, 1→1
y_test_xgb      = le.transform(y_test)
y_train_bal_xgb = le.transform(y_train_balanced)

xgb_param_grid = {
    'n_estimators':    [100, 200],
    'max_depth':       [3, 6, 9],
    'learning_rate':   [0.05, 0.1, 0.2],
    'subsample':       [0.8, 1.0],
    'colsample_bytree':[0.8, 1.0]
}

XGB = GridSearchCV(
    XGBClassifier(random_state=SEED, eval_metric='logloss', use_label_encoder=False),
    xgb_param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
XGB.fit(X_train_scaled, y_train_xgb)
print("XGB best params:", XGB.best_params_)
best_xgb = XGB.best_estimator_

# Balanced variant
XGB_balanced = GridSearchCV(
    XGBClassifier(random_state=SEED, eval_metric='logloss', use_label_encoder=False),
    xgb_param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
XGB_balanced.fit(X_train_balanced, y_train_bal_xgb)
best_xgb_bal = XGB_balanced.best_estimator_

# Evaluate — XGBoost predicts 0/1 so we inverse_transform for consistent labels
def evaluate_xgb(model, X_test, y_test_binary, le, y_test_orig):
    """Evaluate XGBoost; handles 0/1 → -1/1 label remapping."""
    y_pred_bin = model.predict(X_test)
    y_pred     = le.inverse_transform(y_pred_bin)   # back to -1/1
    y_proba    = model.predict_proba(X_test)[:, 1]  # prob of class 1

    return {
        'accuracy':  accuracy_score(y_test_orig, y_pred),
        'precision': precision_score(y_test_orig, y_pred, average='macro'),
        'recall':    recall_score(y_test_orig, y_pred, average='macro'),
        'f1_score':  f1_score(y_test_orig, y_pred, average='macro'),
        'auc_roc':   roc_auc_score(y_test_binary, y_proba)
    }

xgb_eval     = evaluate_xgb(best_xgb,     X_test_scaled, y_test_xgb, le, y_test)
xgb_eval_bal = evaluate_xgb(best_xgb_bal, X_test_scaled, y_test_xgb, le, y_test)

print("XGBoost (Unbalanced):", xgb_eval)
print("XGBoost (Balanced):  ", xgb_eval_bal)

# Predictions for McNemar — back in -1/1 space
pred_xgb     = le.inverse_transform(best_xgb.predict(X_test_scaled))
pred_xgb_bal = le.inverse_transform(best_xgb_bal.predict(X_test_scaled))

# --- UPDATE your results_df to include XGBoost ---
results_extended = [
    {'Model': 'Logistic Regression (Unbalanced)', **lr_eval},
    {'Model': 'Random Forest (Unbalanced)',        **rf_eval},
    {'Model': 'SVM (Unbalanced)',                  **svm_eval},
    {'Model': 'XGBoost (Unbalanced)',              **xgb_eval},
    {'Model': 'Logistic Regression (Balanced)',    **lr_eval_bal},
    {'Model': 'Random Forest (Balanced)',           **rf_eval_bal},
    {'Model': 'SVM (Balanced)',                    **svm_eval_bal},
    {'Model': 'XGBoost (Balanced)',                **xgb_eval_bal},
]
results_df_extended = pd.DataFrame(results_extended)
print(results_df_extended.to_string(index=False))

## Training and Test Performance Comparison

## Statistical Comparison of Classifiers with 10-Fold Cross-Validation on Training Data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from scipy import stats
import warnings

warnings.filterwarnings('ignore') # Suppress warnings from old sklearn/XGBoost versions

# --- Assertion for Data Split Verification ---
# Verify that X_train_scaled and X_test_scaled are distinct and correctly sized
print(f"Shape of X_train_scaled: {X_train_scaled.shape}")
print(f"Shape of X_test_scaled: {X_test_scaled.shape}")

# A more robust check for non-overlapping data (if original indices are available)
# If X_train and X_test were created from pandas DataFrames, check indices.
# Otherwise, we rely on train_test_split's behavior.
# For numpy arrays, a direct check for overlapping values is computationally intensive and not typically done.
# The primary verification is the use of `train_test_split` with `stratify` and `test_size`.

# --- Cross-validation conducted on training partition only (80% of data). ---
# --- Test set (20%) is held out and not involved in this procedure. ---

# 1. Define Classifiers with Fixed Best Hyperparameters

# Assuming these are the best parameters from previous GridSearchCV runs
lr_params = {'C': 0.1, 'penalty': 'elasticnet', 'solver': 'saga', 'l1_ratio': 0.5}
rf_params = {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 2}
# For SVM, use the best parameters found by the expanded grid search (from best_svm_expanded)
svm_params = best_svm_expanded.get_params()
xgb_params = {
    'n_estimators': 200,
    'max_depth': 9,
    'learning_rate': 0.1,
    'subsample': 1.0,
    'colsample_bytree': 0.8,
    'random_state': SEED, # Ensure consistency
    'eval_metric': 'logloss',
    'use_label_encoder': False
}

# Instantiate classifiers
clf_lr = LogisticRegression(random_state=SEED, max_iter=1000, **lr_params)
clf_rf = RandomForestClassifier(random_state=SEED, **rf_params)
clf_svm = SVC(random_state=SEED, probability=True, **{k: svm_params[k] for k in ['C', 'kernel', 'gamma'] if k in svm_params}) # Filter to common SVC params
clf_xgb = XGBClassifier(**xgb_params)

classifiers = {
    'Logistic Regression': clf_lr,
    'Random Forest': clf_rf,
    'SVM': clf_svm,
    'XGBoost': clf_xgb
}

# 2. Run StratifiedKFold Cross-Validation on X_train_scaled, y_train
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

cv_scores = {}
for name, clf in classifiers.items():
    print(f"Performing 10-fold CV for {name}...")
    # XGBoost requires 0/1 labels, so use y_train_xgb for it
    y_for_cv = y_train_xgb if name == 'XGBoost' else y_train
    scores = cross_val_score(clf, X_train_scaled, y_for_cv, cv=skf, scoring='f1_macro', n_jobs=-1)
    cv_scores[name] = scores
    print(f"  {name} F1-macro CV scores: {scores.round(4)}")
    print(f"  {name} Mean F1-macro: {np.mean(scores):.4f} (Std: {np.std(scores):.4f})")

# 3. Cohen's d Function (re-defined to handle edge cases)
def cohens_d(scores_a, scores_b):
    mean_a = np.mean(scores_a)
    mean_b = np.mean(scores_b)
    std_a = np.std(scores_a, ddof=1) # ddof=1 for sample standard deviation
    std_b = np.std(scores_b, ddof=1)

    # Pooled standard deviation (assuming equal variances)
    pooled_std = np.sqrt((std_a**2 + std_b**2) / 2)

    if pooled_std < 1e-9: # If std is essentially zero
        if np.isclose(mean_a, mean_b, atol=1e-9):
            return 0.0
        else:
            return np.inf * np.sign(mean_a - mean_b) # Return +/- infinity
    return (mean_a - mean_b) / pooled_std

# 4. Bootstrap 95% Confidence Intervals for Mean F1 Difference
def bootstrap_ci_diff(scores_a, scores_b, n_resamples=5000, confidence_level=0.95):
    diffs = scores_a - scores_b # Paired differences
    bootstrap_means = []
    n = len(diffs)
    for _ in range(n_resamples):
        resample_diffs = np.random.choice(diffs, n, replace=True)
        bootstrap_means.append(np.mean(resample_diffs))

    lower_bound = np.percentile(bootstrap_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(bootstrap_means, (1 + confidence_level) / 2 * 100)
    return (lower_bound, upper_bound)

# 5. Compile and Print Results Table
comparison_results = []
clf_names = list(classifiers.keys())

print("\n--- Pairwise Classifier Comparison ---")

for i in range(len(clf_names)):
    for j in range(i + 1, len(clf_names)):
        name_a, scores_a = clf_names[i], cv_scores[clf_names[i]]
        name_b, scores_b = clf_names[j], cv_scores[clf_names[j]]

        mean_f1_diff = np.mean(scores_a) - np.mean(scores_b)
        cohens_d_val = cohens_d(scores_a, scores_b)
        ci_low, ci_high = bootstrap_ci_diff(scores_a, scores_b)

        # Handle near-zero variance flag
        std_a = np.std(scores_a)
        std_b = np.std(scores_b)
        cohens_d_note = ""
        if std_a < 1e-6 or std_b < 1e-6:
            cohens_d_note = " (Note: Near-zero variance in CV scores, Cohen's d might be unstable)"

        comparison_results.append({
            'Classifier A': name_a,
            'Classifier B': name_b,
            'Mean F1 Diff (A-B)': f'{mean_f1_diff:.4f}',
            "Cohen's d": f'{cohens_d_val:.4f}' + cohens_d_note,
            'Bootstrap 95% CI': f'[{ci_low:.4f}, {ci_high:.4f}]'
        })

results_df = pd.DataFrame(comparison_results)
print(results_df.to_string(index=False))

# Add a note for interpretation clarity regarding Cohen's d
print("\nInterpretation of Cohen's d (Effect Size):")
print("- 0.2: Small effect (difficult to observe)")
print("- 0.5: Medium effect (visible to the naked eye)")
print("- 0.8: Large effect (grossly perceptible)")
print("For near-zero variance, Bootstrap CI should be the primary interpretation.")

## Framework Overview Figure

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Define colors
COLOR_INPUT_DATA = '#D6EAF8'      # Light Blue
COLOR_PROCESSING = '#F2F3F4'    # Light Grey
COLOR_TRACK_A = '#D4EDDA'       # Light Green
COLOR_TRACK_B = '#FDEBD0'       # Light Orange
COLOR_EVALUATION = '#E8DAEF'    # Light Purple
COLOR_OUTPUT = 'white'          # White with black border

# Define box dimensions and font size
BOX_WIDTH = 0.22  # Relative width of nodes
BOX_HEIGHT = 0.06 # Relative height of nodes
TEXT_FONTSIZE = 7
LINE_WIDTH = 0.7
ARROW_HEAD_SIZE = 0.01 # Adjust based on figure size

# Function to draw a box
def draw_box(ax, center_x, center_y, width, height, text, facecolor, edgecolor='black', linestyle='-', text_color='black', fontsize=TEXT_FONTSIZE, ha='center', va='center'):
    rect = patches.Rectangle((center_x - width / 2, center_y - height / 2), width, height,
                             facecolor=facecolor, edgecolor=edgecolor, linestyle=linestyle, lw=LINE_WIDTH,
                             transform=ax.transAxes) # Use ax.transAxes for relative coordinates
    ax.add_patch(rect)
    ax.text(center_x, center_y, text,
            ha=ha, va=va, fontsize=fontsize, color=text_color,
            transform=ax.transAxes, wrap=True)
    return rect

# Function to draw an arrow
def draw_arrow(ax, start_coords, end_coords, color='black', connectionstyle="arc3,rad=0", arrowstyle='-|>', mutation_scale=7):
    arrow = patches.FancyArrowPatch(start_coords, end_coords,
                                    connectionstyle=connectionstyle,
                                    arrowstyle=arrowstyle, color=color, lw=LINE_WIDTH,
                                    mutation_scale=mutation_scale,
                                    transform=ax.transAxes)
    ax.add_patch(arrow)

# --- Create the figure ---

# Define common y-levels for nodes
y_input = 0.95
y_split = 0.85
y_train_test = 0.75
y_track = 0.62
y_scaler = 0.52
y_gridsearch = 0.42
y_best_models = 0.32
y_eval_stream = 0.20
y_shap = 0.08
y_error_overlap = 0.00
y_output = -0.08

# Define common x-levels
x_center = 0.5
x_left = 0.25
x_right = 0.75

# Create the main figure and axes
fig, ax = plt.subplots(figsize=(7, 10)) # Adjust figsize for better layout in preview
ax.set_xlim(0, 1)
ax.set_ylim(-0.15, 1)
ax.axis('off') # Hide axes

# --- Draw Nodes ---

# 1. Input
node_input = draw_box(ax, x_center, y_input, BOX_WIDTH*1.3, BOX_HEIGHT, "Input: UCI Phishing Websites Dataset\n(11,055 instances, 30 features)", COLOR_INPUT_DATA)

# 2. Stratified Split
node_split = draw_box(ax, x_center, y_split, BOX_WIDTH, BOX_HEIGHT, "Stratified 80/20 Train-Test Split", COLOR_PROCESSING)

# 3. Training Set & Test Set
node_train_set = draw_box(ax, x_left, y_train_test, BOX_WIDTH, BOX_HEIGHT, "Training Set\n(8,844 instances)", COLOR_INPUT_DATA)
node_test_set = draw_box(ax, x_right, y_train_test, BOX_WIDTH, BOX_HEIGHT, "Test Set\n(2,211 instances) [LOCKED]", COLOR_INPUT_DATA)

# Note Box for Test Set
note_text = "Test set is locked after initial split and\nnever used in hyperparameter tuning or cross-validation."
node_note = draw_box(ax, x_right, y_train_test - BOX_HEIGHT - 0.05, BOX_WIDTH*0.9, BOX_HEIGHT*1.2, note_text, 'white', edgecolor='red', text_color='red', fontsize=6)

# 4. Two Parallel Training Tracks
node_track_a = draw_box(ax, x_left, y_track, BOX_WIDTH, BOX_HEIGHT, "Track A: Original (Imbalanced) Training Data", COLOR_TRACK_A)
node_track_b = draw_box(ax, x_right, y_track, BOX_WIDTH, BOX_HEIGHT, "Track B: SMOTE-Balanced Training Data", COLOR_TRACK_B)

# 5. StandardScaler
node_scaler_a = draw_box(ax, x_left, y_scaler, BOX_WIDTH, BOX_HEIGHT*1.1, "StandardScaler\n(fit on training data only, transform both)", COLOR_PROCESSING)
node_scaler_b = draw_box(ax, x_right, y_scaler, BOX_WIDTH, BOX_HEIGHT*1.1, "StandardScaler\n(fit on training data only, transform both)", COLOR_PROCESSING)

# 6. GridSearchCV
node_gridsearch_a = draw_box(ax, x_left, y_gridsearch, BOX_WIDTH*1.1, BOX_HEIGHT*1.1, "GridSearchCV (5-fold CV, f1_macro)\nClassifiers: LR, RF, SVM, XGBoost", COLOR_PROCESSING)
node_gridsearch_b = draw_box(ax, x_right, y_gridsearch, BOX_WIDTH*1.1, BOX_HEIGHT*1.1, "GridSearchCV (5-fold CV, f1_macro)\nClassifiers: LR, RF, SVM, XGBoost", COLOR_PROCESSING)

# 7. Best-parameter Models
node_best_models = draw_box(ax, x_center, y_best_models, BOX_WIDTH*1.5, BOX_HEIGHT, "Best-parameter models trained on full training data (per track)", COLOR_PROCESSING)

# 8. Evaluation Streams
node_eval_stream1 = draw_box(ax, x_left, y_eval_stream, BOX_WIDTH*1.1, BOX_HEIGHT*1.1, "Stream 1: Evaluate on Test Set\n(Classification Metrics, McNemar's Test)", COLOR_EVALUATION)
node_eval_stream2 = draw_box(ax, x_right, y_eval_stream, BOX_WIDTH*1.1, BOX_HEIGHT*1.1, "Stream 2: 10-fold CV on Training Partition\n(Cohen's d, Bootstrap CIs)", COLOR_EVALUATION)

# 9. SHAP Analysis
node_shap = draw_box(ax, x_center, y_shap, BOX_WIDTH*1.3, BOX_HEIGHT, "SHAP Analysis (on Test Set predictions, all classifiers)", COLOR_PROCESSING)

# 10. Error Overlap Analysis
node_error_overlap = draw_box(ax, x_center, y_error_overlap, BOX_WIDTH*1.3, BOX_HEIGHT, "Error Overlap Analysis (on Test Set predictions)", COLOR_PROCESSING)

# 11. Output Nodes
node_output_tables = draw_box(ax, x_center - BOX_WIDTH*0.4, y_output, BOX_WIDTH*0.5, BOX_HEIGHT*0.7, "Tables", COLOR_OUTPUT)
node_output_figures = draw_box(ax, x_center + BOX_WIDTH*0.4, y_output, BOX_WIDTH*0.5, BOX_HEIGHT*0.7, "Figures", COLOR_OUTPUT)

# --- Draw Arrows ---

# Input -> Split
draw_arrow(ax, (x_center, y_input - BOX_HEIGHT/2), (x_center, y_split + BOX_HEIGHT/2))

# Split -> Training Set
draw_arrow(ax, (x_center, y_split - BOX_HEIGHT/2), (x_left, y_train_test + BOX_HEIGHT/2))

# Split -> Test Set
draw_arrow(ax, (x_center, y_split - BOX_HEIGHT/2), (x_right, y_train_test + BOX_HEIGHT/2))

# Training Set -> Track A
draw_arrow(ax, (x_left, y_train_test - BOX_HEIGHT/2), (x_left, y_track + BOX_HEIGHT/2))

# Training Set -> Track B (diverging from left to right)
draw_arrow(ax, (x_left, y_train_test - BOX_HEIGHT/2), (x_right, y_track + BOX_HEIGHT/2), connectionstyle="arc3,rad=0.3")

# Track A -> StandardScaler A
draw_arrow(ax, (x_left, y_track - BOX_HEIGHT/2), (x_left, y_scaler + BOX_HEIGHT/2))

# Track B -> StandardScaler B
draw_arrow(ax, (x_right, y_track - BOX_HEIGHT/2), (x_right, y_scaler + BOX_HEIGHT/2))

# StandardScaler A -> GridSearchCV A
draw_arrow(ax, (x_left, y_scaler - BOX_HEIGHT/2), (x_left, y_gridsearch + BOX_HEIGHT/2))

# StandardScaler B -> GridSearchCV B
draw_arrow(ax, (x_right, y_scaler - BOX_HEIGHT/2), (x_right, y_gridsearch + BOX_HEIGHT/2))

# GridSearchCV A -> Best Models
draw_arrow(ax, (x_left, y_gridsearch - BOX_HEIGHT/2), (x_center, y_best_models + BOX_HEIGHT/2), connectionstyle="arc3,rad=0.2")

# GridSearchCV B -> Best Models
draw_arrow(ax, (x_right, y_gridsearch - BOX_HEIGHT/2), (x_center, y_best_models + BOX_HEIGHT/2), connectionstyle="arc3,rad=-0.2")

# Best Models -> Evaluation Stream 1 (diverging to left)
draw_arrow(ax, (x_center, y_best_models - BOX_HEIGHT/2), (x_left, y_eval_stream + BOX_HEIGHT/2), connectionstyle="arc3,rad=0.2")

# Best Models -> Evaluation Stream 2 (diverging to right)
draw_arrow(ax, (x_center, y_best_models - BOX_HEIGHT/2), (x_right, y_eval_stream + BOX_HEIGHT/2), connectionstyle="arc3,rad=-0.2")

# Evaluation Stream 1 -> SHAP Analysis (converging to center)
draw_arrow(ax, (x_left, y_eval_stream - BOX_HEIGHT/2), (x_center, y_shap + BOX_HEIGHT/2), connectionstyle="arc3,rad=0.2")

# Evaluation Stream 2 -> SHAP Analysis (converging to center)
draw_arrow(ax, (x_right, y_eval_stream - BOX_HEIGHT/2), (x_center, y_shap + BOX_HEIGHT/2), connectionstyle="arc3,rad=-0.2")

# SHAP Analysis -> Error Overlap Analysis
draw_arrow(ax, (x_center, y_shap - BOX_HEIGHT/2), (x_center, y_error_overlap + BOX_HEIGHT/2))

# Error Overlap Analysis -> Output Tables (diverging left)
draw_arrow(ax, (x_center, y_error_overlap - BOX_HEIGHT/2), (x_center - BOX_WIDTH*0.4, y_output + BOX_HEIGHT*0.35), connectionstyle="arc3,rad=0.1")

# Error Overlap Analysis -> Output Figures (diverging right)
draw_arrow(ax, (x_center, y_error_overlap - BOX_HEIGHT/2), (x_center + BOX_WIDTH*0.4, y_output + BOX_HEIGHT*0.35), connectionstyle="arc3,rad=-0.1")

# --- Final adjustments and saving ---
plt.suptitle('Figure 1: Machine Learning Benchmarking Framework', fontsize=12, y=0.99) # Add a figure title
plt.tight_layout(rect=[0, 0, 1, 0.98]) # Adjust layout to prevent suptitle overlap

# Save in single-column width (3.5 inches)
fig.set_size_inches(3.5, 7) # Width, Height
plt.savefig('framework_figure_single_column.pdf', dpi=300, bbox_inches='tight')

# Save in double-column width (7 inches)
fig.set_size_inches(7, 10) # Reset to a more appropriate aspect ratio for double column
plt.savefig('framework_figure.pdf', dpi=300, bbox_inches='tight')

plt.show()

## Hyperparameter Tuning Summary Table

In [ ]:
import pandas as pd

# Helper function to format search range for display
def format_range(param_values):
    if isinstance(param_values, list):
        # Convert non-numeric types to string for consistent display
        formatted_values = [str(x) for x in param_values]
        return '{' + ', '.join(formatted_values) + '}'
    return str(param_values)

# Helper function to check boundary and add flag
def check_boundary(value, param_range):
    if not isinstance(param_range, list) or len(param_range) < 2:
        return str(value) # Not a list, or list too short to have boundaries

    # Filter out non-numeric values for min/max comparison if value is numeric
    numeric_range = [x for x in param_range if isinstance(x, (int, float))]

    if isinstance(value, (int, float)):
        if not numeric_range: # No numeric values in range, cannot check numeric boundary
            return str(value)
        if value == min(numeric_range) or value == max(numeric_range):
            return f"{value}†"
    else: # Handle categorical or string values
        if value == param_range[0] or value == param_range[-1]:
            return f"{value}†"
    return str(value)

data = []

# --- 1. Logistic Regression Parameters (from GSCV and GSCV_balanced) ---
lr_best_unbal = GSCV.best_params_
lr_best_bal = GSCV_balanced.best_params_

lr_params_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2', 'elasticnet'],
    'solver': ['liblinear', 'saga', 'newton-cg', 'lbfgs', 'sag'], # Union of all solvers used
    'l1_ratio': [0.1, 0.5, 0.9]
}

# C, penalty, solver always present
for param in ['C', 'penalty', 'solver']:
    data.append({
        'Classifier': 'Logistic Regression',
        'Hyperparameter': param,
        'Search Range': format_range(lr_params_grid[param]),
        'Optimal (Unbalanced)': check_boundary(lr_best_unbal.get(param, 'N/A'), lr_params_grid[param]),
        'Optimal (SMOTE)': check_boundary(lr_best_bal.get(param, 'N/A'), lr_params_grid[param])
    })
# l1_ratio only for elasticnet penalty
data.append({
    'Classifier': 'Logistic Regression',
    'Hyperparameter': 'l1_ratio',
    'Search Range': format_range(lr_params_grid['l1_ratio']),
    'Optimal (Unbalanced)': check_boundary(lr_best_unbal.get('l1_ratio', 'N/A'), lr_params_grid['l1_ratio']) if lr_best_unbal.get('penalty') == 'elasticnet' else 'N/A',
    'Optimal (SMOTE)': check_boundary(lr_best_bal.get('l1_ratio', 'N/A'), lr_params_grid['l1_ratio']) if lr_best_bal.get('penalty') == 'elasticnet' else 'N/A'
})

# --- 2. Random Forest Parameters (from RandomForest and RandomForest_balanced) ---
rf_best_unbal = RandomForest.best_params_
rf_best_bal = RandomForest_balanced.best_params_

rf_params_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}

for param, s_range in rf_params_grid.items():
    data.append({
        'Classifier': 'Random Forest',
        'Hyperparameter': param,
        'Search Range': format_range(s_range),
        'Optimal (Unbalanced)': check_boundary(rf_best_unbal.get(param), s_range),
        'Optimal (SMOTE)': check_boundary(rf_best_bal.get(param), s_range)
    })

# --- 3. SVM Parameters (from SVM_expanded and SVM_balanced_expanded) ---
svm_best_unbal = SVM_expanded.best_params_ # Using the expanded grid results
svm_best_bal = SVM_balanced_expanded.best_params_ # Using the expanded grid results

svm_params_grid = {
    'C': [0.01, 0.1, 1, 10, 100, 1000],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1.0] # Union of gamma ranges
}

for param, s_range in svm_params_grid.items():
    # Special handling for gamma to correctly check for numerical boundaries if value is numeric
    if param == 'gamma':
        numerical_gamma_range = [x for x in s_range if isinstance(x, (int, float))]
        data.append({
            'Classifier': 'SVM',
            'Hyperparameter': param,
            'Search Range': format_range(s_range),
            'Optimal (Unbalanced)': check_boundary(svm_best_unbal.get(param), numerical_gamma_range if isinstance(svm_best_unbal.get(param), (int, float)) else s_range),
            'Optimal (SMOTE)': check_boundary(svm_best_bal.get(param), numerical_gamma_range if isinstance(svm_best_bal.get(param), (int, float)) else s_range)
        })
    else:
        data.append({
            'Classifier': 'SVM',
            'Hyperparameter': param,
            'Search Range': format_range(s_range),
            'Optimal (Unbalanced)': check_boundary(svm_best_unbal.get(param), s_range),
            'Optimal (SMOTE)': check_boundary(svm_best_bal.get(param), s_range)
        })

# --- 4. XGBoost Parameters (from XGB and XGB_balanced) ---
xgb_best_unbal = XGB.best_params_
xgb_best_bal = XGB_balanced.best_params_

xgb_params_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

for param, s_range in xgb_params_grid.items():
    data.append({
        'Classifier': 'XGBoost',
        'Hyperparameter': param,
        'Search Range': format_range(s_range),
        'Optimal (Unbalanced)': check_boundary(xgb_best_unbal.get(param), s_range),
        'Optimal (SMOTE)': check_boundary(xgb_best_bal.get(param), s_range)
    })

# --- Create DataFrame and Format for LaTeX ---
df_hyperparams = pd.DataFrame(data)

# For multirow in LaTeX, set 'Classifier' as index and keep other columns
df_hyperparams_latex = df_hyperparams.set_index(['Classifier', 'Hyperparameter'])

# Generate LaTeX string
latex_output = df_hyperparams_latex.to_latex(
    index=True,
    caption="Hyperparameter search ranges and optimal values for all classifiers under both experimental conditions. † indicates boundary selection.",
    label="tab:hyperparams",
    header=['Hyperparameter', 'Search Range', 'Optimal (Unbalanced)', 'Optimal (SMOTE)'],
    escape=False, # Crucial for the '†' symbol
    column_format='llccc',
    multirow=True,
    longtable=False,
    position='h!' # Add position specifier for LaTeX table
)

# Add booktabs commands and footnote for LaTeX formatting
latex_output = latex_output.replace('\toprule', '\toprule\n\\addlinespace[0.5em]')
latex_output = latex_output.replace('\midrule', '\midrule\n\\addlinespace[0.5em]')
latex_output = latex_output.replace('\bottomrule', '\addlinespace[0.5em]\n\\bottomrule')

# Add footnote explanation for † symbol
latex_output = latex_output.replace(
    '\end{tabular}',
    '\end{tabular}\n\raggedright\par\small † Optimal value lies at grid boundary; expanded search recommended.'
)

# Print the LaTeX output to console
print("\n--- LaTeX Table Output ---")
print(latex_output)

# Save to file
with open('hyperparameter_table.tex', 'w') as f:
    f.write(latex_output)

# Print a plain-text summary for verification
print("\n--- Plain-Text Summary for Verification ---")
print(df_hyperparams.to_string(index=False))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Gather Training Metrics ---

# Unbalanced Data Training Evaluation
lr_train_eval = evaluate_model(best_lr, X_train_scaled, y_train)
rf_train_eval = evaluate_model(best_rf, X_train_scaled, y_train)
svm_train_eval_expanded = evaluate_model(best_svm_expanded, X_train_scaled, y_train)
xgb_train_eval = evaluate_xgb(best_xgb, X_train_scaled, y_train_xgb, le, y_train)

# Balanced Data Training Evaluation
lr_train_eval_bal = evaluate_model(best_lr_bal, X_train_balanced, y_train_balanced)
rf_train_eval_bal = evaluate_model(best_rf_bal, X_train_balanced, y_train_balanced)
svm_train_eval_bal_expanded = evaluate_model(best_svm_bal_expanded, X_train_balanced, y_train_balanced)
xgb_train_eval_bal = evaluate_xgb(best_xgb_bal, X_train_balanced, y_train_bal_xgb, le, y_train_balanced)

# --- Compile Results into DataFrame ---
comparison_data = [
    {
        'Model': 'Logistic Regression',
        'Condition': 'Unbalanced',
        'Train_Accuracy': lr_train_eval['accuracy'],
        'Train_F1': lr_train_eval['f1_score'],
        'Test_Accuracy': lr_eval['accuracy'],
        'Test_F1': lr_eval['f1_score']
    },
    {
        'Model': 'Random Forest',
        'Condition': 'Unbalanced',
        'Train_Accuracy': rf_train_eval['accuracy'],
        'Train_F1': rf_train_eval['f1_score'],
        'Test_Accuracy': rf_eval['accuracy'],
        'Test_F1': rf_eval['f1_score']
    },
    {
        'Model': 'SVM',
        'Condition': 'Unbalanced',
        'Train_Accuracy': svm_train_eval_expanded['accuracy'],
        'Train_F1': svm_train_eval_expanded['f1_score'],
        'Test_Accuracy': svm_eval_expanded['accuracy'],
        'Test_F1': svm_eval_expanded['f1_score']
    },
    {
        'Model': 'XGBoost',
        'Condition': 'Unbalanced',
        'Train_Accuracy': xgb_train_eval['accuracy'],
        'Train_F1': xgb_train_eval['f1_score'],
        'Test_Accuracy': xgb_eval['accuracy'],
        'Test_F1': xgb_eval['f1_score']
    },
    {
        'Model': 'Logistic Regression',
        'Condition': 'SMOTE-Balanced',
        'Train_Accuracy': lr_train_eval_bal['accuracy'],
        'Train_F1': lr_train_eval_bal['f1_score'],
        'Test_Accuracy': lr_eval_bal['accuracy'],
        'Test_F1': lr_eval_bal['f1_score']
    },
    {
        'Model': 'Random Forest',
        'Condition': 'SMOTE-Balanced',
        'Train_Accuracy': rf_train_eval_bal['accuracy'],
        'Train_F1': rf_train_eval_bal['f1_score'],
        'Test_Accuracy': rf_eval_bal['accuracy'],
        'Test_F1': rf_eval_bal['f1_score']
    },
    {
        'Model': 'SVM',
        'Condition': 'SMOTE-Balanced',
        'Train_Accuracy': svm_train_eval_bal_expanded['accuracy'],
        'Train_F1': svm_train_eval_bal_expanded['f1_score'],
        'Test_Accuracy': svm_eval_bal_expanded['accuracy'],
        'Test_F1': svm_eval_bal_expanded['f1_score']
    },
    {
        'Model': 'XGBoost',
        'Condition': 'SMOTE-Balanced',
        'Train_Accuracy': xgb_train_eval_bal['accuracy'],
        'Train_F1': xgb_train_eval_bal['f1_score'],
        'Test_Accuracy': xgb_eval_bal['accuracy'],
        'Test_F1': xgb_eval_bal['f1_score']
    }
]

comparison_df = pd.DataFrame(comparison_data)
comparison_df['F1_Gap'] = comparison_df['Train_F1'] - comparison_df['Test_F1']

print("--- Training and Test Performance Comparison ---")
print(comparison_df.to_string(index=False, float_format="%.4f"))

In [ ]:
# --- Overfitting Warning Flag ---
print("\n--- Overfitting Warnings (F1_Gap > 0.02) ---")
overfitting_models = comparison_df[comparison_df['F1_Gap'] > 0.02]

if not overfitting_models.empty:
    for index, row in overfitting_models.iterrows():
        print(f"WARNING: Model '{row['Model']}' with '{row['Condition']}' data shows a significant F1_Gap ({row['F1_Gap']:.4f}), suggesting possible overfitting.")
else:
    print("No models show a significant F1_Gap (> 0.02) indicative of overfitting.")

In [ ]:
# --- Export to LaTeX Table ---

# Create a copy to modify for LaTeX formatting without altering original DataFrame
comparison_df_latex = comparison_df.copy()

# Find the row with the highest Test_F1 for each condition and bold it
latex_table_rows = []
for condition in ['Unbalanced', 'SMOTE-Balanced']:
    condition_df = comparison_df_latex[comparison_df_latex['Condition'] == condition]
    if not condition_df.empty:
        max_f1_idx = condition_df['Test_F1'].idxmax()
        for idx, row in condition_df.iterrows():
            row_str_list = [
                row['Model'],
                row['Condition'],
                f"{row['Train_Accuracy']:.4f}",
                f"{row['Train_F1']:.4f}",
                f"{row['Test_Accuracy']:.4f}"
            ]
            if idx == max_f1_idx:
                row_str_list.append(f"\\textbf{{{row['Test_F1']:.4f}}}")
            else:
                row_str_list.append(f"{row['Test_F1']:.4f}")
            row_str_list.append(f"{row['F1_Gap']:.4f}")
            latex_table_rows.append(' & '.join(row_str_list) + ' \\\\')
    if condition == 'Unbalanced':
        latex_table_rows.append('\\midrule')

latex_output = """
\\begin{{table}}[htbp]
\\centering
\\caption{{Training and test macro F1 comparison across classifiers and conditions. F1 Gap = Train F1 - Test F1; values near zero indicate consistent generalization.}}
\\label{{tab:train_test_comparison}}
\\begin{{tabular}}{{llrrrrr}}
\\toprule
Model & Condition & Train\\_Accuracy & Train\\_F1 & Test\\_Accuracy & Test\\_F1 & F1\\_Gap \\\\
\\midrule
{}
\\bottomrule
\\end{{tabular}}
\\end{{table}}
""".format('\n'.join(latex_table_rows))

print("\n--- LaTeX Table Output ---")
print(latex_output)

In [ ]:
# --- Generate Grouped Bar Chart ---

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle('Training vs. Test Macro F1 Score Comparison', fontsize=16)

# Unbalanced Condition
unbalanced_df = comparison_df[comparison_df['Condition'] == 'Unbalanced']
metrics_unbalanced = unbalanced_df[['Model', 'Train_F1', 'Test_F1']].set_index('Model')
metrics_unbalanced.plot(kind='bar', ax=axes[0], colormap='viridis')
axes[0].set_title('Unbalanced Data')
axes[0].set_ylabel('Macro F1 Score')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Metric')
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# SMOTE-Balanced Condition
balanced_df = comparison_df[comparison_df['Condition'] == 'SMOTE-Balanced']
metrics_balanced = balanced_df[['Model', 'Train_F1', 'Test_F1']].set_index('Model')
metrics_balanced.plot(kind='bar', ax=axes[1], colormap='plasma')
axes[1].set_title('SMOTE-Balanced Data')
axes[1].set_ylabel('Macro F1 Score')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Metric')
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap
plt.savefig('train_test_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!pip install shap
import shap
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
from scipy.stats import spearmanr
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
shap.initjs()

feature_names = list(X.columns)

# --- 2a. Random Forest — TreeExplainer ---
explainer_rf   = shap.TreeExplainer(best_rf)
shap_values_rf = explainer_rf.shap_values(X_test_scaled)

# Fix: Ensure sv_rf is 2D (samples, features) for class 1
if isinstance(shap_values_rf, list):
    sv_rf = shap_values_rf[1]
elif len(shap_values_rf.shape) == 3:
    sv_rf = shap_values_rf[:, :, 1]
else:
    sv_rf = shap_values_rf

# --- 2b. XGBoost — TreeExplainer ---
explainer_xgb   = shap.TreeExplainer(best_xgb)
sv_xgb = explainer_xgb.shap_values(X_test_scaled)

# --- 2c. Logistic Regression — LinearExplainer ---
background_lr   = shap.maskers.Independent(X_train_scaled, max_samples=500)
explainer_lr    = shap.LinearExplainer(best_lr, background_lr)
sv_lr = explainer_lr.shap_values(X_test_scaled)

# --- 2d. SVM — KernelExplainer ---
print("Computing SVM SHAP values (approx. 2-3 mins)...")
background_svm  = shap.kmeans(X_train_scaled, 50)
explainer_svm   = shap.KernelExplainer(best_svm.predict_proba, background_svm)
np.random.seed(SEED)
idx_sample      = np.random.choice(len(X_test_scaled), size=200, replace=False)
X_test_sample   = X_test_scaled[idx_sample]
shap_values_svm = explainer_svm.shap_values(X_test_sample)

# Fix: Handle all possible SHAP output formats for SVM
if isinstance(shap_values_svm, list):
    # Old SHAP format: list of arrays
    sv_svm_sample = shap_values_svm[1]
elif len(shap_values_svm.shape) == 3:
    # New SHAP format: (samples, features, classes)
    sv_svm_sample = shap_values_svm[:, :, 1]
else:
    # Already 2D (samples, features)
    sv_svm_sample = shap_values_svm

print("SVM SHAP done.")

# Verification of shapes
print(f"RF shape:  {sv_rf.shape}")
print(f"XGB shape: {sv_xgb.shape}")
print(f"LR shape:  {sv_lr.shape}")
print(f"SVM shape: {sv_svm_sample.shape}")

# ---- Importance Calculation and Plotting ----
def mean_abs_shap(sv, feature_names):
    # Ensure we are taking the mean across samples to get a 1D vector of length 31
    return pd.Series(np.abs(sv).mean(axis=0), index=feature_names)

shap_importance_rf  = mean_abs_shap(sv_rf,  feature_names)
shap_importance_xgb = mean_abs_shap(sv_xgb, feature_names)
shap_importance_lr  = mean_abs_shap(sv_lr,  feature_names)
shap_importance_svm = mean_abs_shap(sv_svm_sample, feature_names)

# Build comparison and plot
top15 = shap_importance_rf.nlargest(15).index.tolist()
shap_comparison = pd.DataFrame({
    'Random Forest': shap_importance_rf[top15],
    'XGBoost':       shap_importance_xgb[top15],
    'Logistic Reg':  shap_importance_lr[top15],
    'SVM':           shap_importance_svm[top15]
}, index=top15)

fig, ax = plt.subplots(figsize=(10, 7))
shap_comparison.plot(kind='barh', ax=ax, width=0.75)
ax.invert_yaxis()
ax.set_title('Top 15 Features — Mean |SHAP| Comparison')
plt.tight_layout()
plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt

# ---- FIGURE 5: Global SHAP Summary — Beeswarm (Random Forest) ----
plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv_rf,
    X_test_scaled,
    feature_names=feature_names,
    max_display=15,
    show=False,
    plot_type='dot'
)
plt.title('Figure 5: SHAP Summary Plot — Random Forest (Phishing Class)', fontsize=13)
plt.tight_layout()
plt.savefig('fig5_shap_summary_rf.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ---- FIGURE 6: Global SHAP Bar Chart Comparison — all 4 models ----
fig, ax = plt.subplots(figsize=(10, 8))
shap_comparison.plot(kind='barh', ax=ax, width=0.8, alpha=0.9)
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_title('Figure 6: Model-Agnostic Feature Importance (Mean |SHAP|)\nTop 15 Features — All Classifiers', fontsize=13)
ax.legend(title='Classifier', loc='lower right')
plt.tight_layout()
plt.savefig('fig6_shap_comparison_all_models.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt

# ---- FIGURE 7: SHAP Dependence Plot — SSLfinal_State ----
ssl_idx = feature_names.index('SSLfinal_State')
anchor_idx = feature_names.index('URL_of_Anchor')

plt.figure(figsize=(8, 5))
shap.dependence_plot(
    ssl_idx,
    sv_rf,
    X_test_scaled,
    feature_names=feature_names,
    interaction_index=anchor_idx,
    show=False,
    ax=plt.gca()
)
plt.title('Figure 7: SHAP Dependence Plot — SSLfinal_State (RF)', fontsize=13)
plt.tight_layout()
plt.savefig('fig7_shap_dependence_ssl.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ---- FIGURE 8: Error Overlap Analysis ----
y_test_arr = y_test.values

correct_rf  = (pred_rf  == y_test_arr)
correct_svm = (pred_svm == y_test_arr)
correct_xgb = (pred_xgb == y_test_arr)
correct_lr  = (pred_lr  == y_test_arr)

error_counts = {
    'LR only':         int((~correct_lr & correct_rf & correct_svm & correct_xgb).sum()),
    'RF only':         int((~correct_rf & correct_lr & correct_svm & correct_xgb).sum()),
    'SVM only':        int((~correct_svm & correct_lr & correct_rf & correct_xgb).sum()),
    'XGB only':        int((~correct_xgb & correct_lr & correct_rf & correct_svm).sum()),
    'RF+SVM':          int((~correct_rf & ~correct_svm & correct_lr & correct_xgb).sum()),
    'All wrong':       int((~correct_lr & ~correct_rf & ~correct_svm & ~correct_xgb).sum()),
}

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=list(error_counts.keys()), y=list(error_counts.values()), palette='magma')
plt.ylabel('Number of Instances')
plt.title('Figure 8: Error Overlap Analysis — Classifier Disagreement Patterns')

for p in ax.patches:
    ax.annotate(format(p.get_height(), '.0f'),
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha = 'center', va = 'center',
                xytext = (0, 9),
                textcoords = 'offset points')

plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('fig8_error_overlap.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import itertools
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

# Generate missing predictions for balanced models
pred_lr_bal = best_lr_bal.predict(X_test_scaled)
pred_rf_bal = best_rf_bal.predict(X_test_scaled)
pred_svm_bal = best_svm_bal.predict(X_test_scaled)

# Create a dictionary of all predictions
all_predictions = {
    'LR_Unbalanced': pred_lr,
    'RF_Unbalanced': pred_rf,
    'SVM_Unbalanced': pred_svm,
    'LR_Balanced': pred_lr_bal,
    'RF_Balanced': pred_rf_bal,
    'SVM_Balanced': pred_svm_bal
}

# Iterate through all unique pairs of classifiers
for (name_a, preds_a), (name_b, preds_b) in itertools.combinations(all_predictions.items(), 2):
    print(f"\n--- Comparing {name_a} and {name_b} ---")

    # Create boolean arrays for correctness
    correct_a = (preds_a == y_test.values)
    correct_b = (preds_b == y_test.values)

    # Build the 4 cells of the contingency table
    n_00 = ((correct_a == True) & (correct_b == True)).sum()
    n_01 = ((correct_a == True) & (correct_b == False)).sum()
    n_10 = ((correct_a == False) & (correct_b == True)).sum()
    n_11 = ((correct_a == False) & (correct_b == False)).sum()

    # Create the contingency table
    contingency_table = [[n_00, n_01],
                         [n_10, n_11]]

    print(f"Contingency Table:\n{np.array(contingency_table)}")

    # Perform McNemar's test
    result = mcnemar(contingency_table, exact=False)
    print(f"McNemar's Test Result:\nStatistic: {result.statistic:.4f}, p-value: {result.pvalue:.4f}")

    alpha = 0.05
    if result.pvalue < alpha:
        print("Conclusion: Statistically significant difference between the two classifiers (reject H0).")
    else:
        print("Conclusion: No statistically significant difference between the two classifiers (fail to reject H0).")

In [ ]:
print(results_df['Model'].unique())

In [ ]:
# Define the cohens_d function
def cohens_d(scores_a, scores_b):
    mean_a = np.mean(scores_a)
    mean_b = np.mean(scores_b)

    std_a = np.std(scores_a, ddof=1)
    std_b = np.std(scores_b, ddof=1)

    # Handle cases where std_a or std_b might be zero or extremely small
    pooled_std = np.sqrt((std_a**2 + std_b**2) / 2)

    if pooled_std < 1e-9: # A very small number close to zero
        # If pooled_std is effectively zero, and means are different, d is very large/infinite.
        # If means are also identical, d is zero.
        if np.isclose(mean_a, mean_b):
            return 0.0
        else:
            # Return a very large number to indicate extreme effect if std is zero but means differ
            return np.inf if (mean_a - mean_b) > 0 else -np.inf

    d = (mean_a - mean_b) / pooled_std
    return d

# Perform 10-fold cross-validation on the training set for each classifier
# Use X_train_scaled and y_train as best_lr, best_rf, best_svm were found using this data.
scores_lr = cross_val_score(best_lr, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)
scores_rf = cross_val_score(best_rf, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)
scores_svm = cross_val_score(best_svm, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)

print("10-fold cross-validation F1-macro scores:")
print(f"Logistic Regression (Unbalanced): {scores_lr}")
print(f"Random Forest (Unbalanced): {scores_rf}")
print(f"SVM (Unbalanced): {scores_svm}")

# Compute Cohen's d for each pair
d_lr_rf = cohens_d(scores_lr, scores_rf)
d_lr_svm = cohens_d(scores_lr, scores_svm)
d_rf_svm = cohens_d(scores_rf, scores_svm)

print("\n--- Cohen's d Values and Interpretation ---")
print("\nCohen's d: A measure of effect size. General guidelines:")
print("- Small effect: d = 0.2")
print("- Medium effect: d = 0.5")
print("- Large effect: d = 0.8")

# Interpretation function
def interpret_cohens_d(d_value, name_a, name_b):
    print(f"\n{name_a} vs {name_b}:")
    print(f"  Cohen's d: {d_value:.4f}")
    if np.isinf(d_value):
        print("  Interpretation: Infinite effect size due to zero variance in scores. This indicates a severe issue with one of the score sets (likely constant scores).")
    elif abs(d_value) >= 0.8:
        print("  Interpretation: Large effect size.")
    elif abs(d_value) >= 0.5:
        print("  Interpretation: Medium effect size.")
    elif abs(d_value) >= 0.2:
        print("  Interpretation: Small effect size.")
    else:
        print("  Interpretation: Negligible effect size.")

interpret_cohens_d(d_lr_rf, "Logistic Regression (Unbalanced)", "Random Forest (Unbalanced)")
interpret_cohens_d(d_lr_svm, "Logistic Regression (Unbalanced)", "SVM (Unbalanced)")
interpret_cohens_d(d_rf_svm, "Random Forest (Unbalanced)", "SVM (Unbalanced)")

print("\n--- Important Note on SVM Scores ---")
print("The cross-validation F1-macro scores for SVM (Unbalanced) are very low and nearly constant across folds.")
print("This results in an extremely small standard deviation for SVM scores, leading to exceptionally large Cohen's d values")
print("when compared against other classifiers, and potentially infinite if the std was exactly zero. This suggests an issue")
print("with the cross-validation setup or the SVM model's performance on the training folds, making the Cohen's d")
print("interpretation for pairs involving SVM unreliable in terms of magnitude.")
print("It is unusual for best_svm, which achieved a high F1-macro score in GridSearchCV, to perform so poorly and consistently low")
print("during cross_val_score, suggesting a potential discrepancy in how the metrics or data splits are handled.")

In [ ]:
import itertools
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.model_selection import cross_val_score
import numpy as np

# Define the cohens_d function
def cohens_d(scores_a, scores_b):
    mean_a = np.mean(scores_a)
    mean_b = np.mean(scores_b)

    std_a = np.std(scores_a, ddof=1)
    std_b = np.std(scores_b, ddof=1)

    pooled_std = np.sqrt((std_a**2 + std_b**2) / 2)

    if pooled_std < 1e-9: # A very small number close to zero
        if np.isclose(mean_a, mean_b):
            return 0.0
        else:
            return np.inf if (mean_a - mean_b) > 0 else -np.inf

    d = (mean_a - mean_b) / pooled_std
    return d

# Perform 10-fold cross-validation on the training set for each classifier
# Use X_train_scaled and y_train for unbalanced models
scores_lr_unbal = cross_val_score(best_lr, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)
scores_rf_unbal = cross_val_score(best_rf, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)
scores_svm_unbal = cross_val_score(best_svm, X_train_scaled, y_train, scoring='f1_macro', cv=10, n_jobs=-1)

# Use X_train_balanced and y_train_balanced for balanced models
scores_lr_bal = cross_val_score(best_lr_bal, X_train_balanced, y_train_balanced, scoring='f1_macro', cv=10, n_jobs=-1)
scores_rf_bal = cross_val_score(best_rf_bal, X_train_balanced, y_train_balanced, scoring='f1_macro', cv=10, n_jobs=-1)
scores_svm_bal = cross_val_score(best_svm_bal, X_train_balanced, y_train_balanced, scoring='f1_macro', cv=10, n_jobs=-1)

all_scores = {
    'LR_Unbalanced': scores_lr_unbal,
    'RF_Unbalanced': scores_rf_unbal,
    'SVM_Unbalanced': scores_svm_unbal,
    'LR_Balanced': scores_lr_bal,
    'RF_Balanced': scores_rf_bal,
    'SVM_Balanced': scores_svm_bal
}

# Prepare predictions for McNemar's test (predictions on X_test_scaled)
all_predictions = {
    'LR_Unbalanced': best_lr.predict(X_test_scaled),
    'RF_Unbalanced': best_rf.predict(X_test_scaled),
    'SVM_Unbalanced': best_svm.predict(X_test_scaled),
    'LR_Balanced': best_lr_bal.predict(X_test_scaled),
    'RF_Balanced': best_rf_bal.predict(X_test_scaled),
    'SVM_Balanced': best_svm_bal.predict(X_test_scaled)
}

combined_results = []
alpha = 0.05

# Iterate through all unique pairs of classifiers
for (name_a, scores_a), (name_b, scores_b) in itertools.combinations(all_scores.items(), 2):
    # Cohen's d calculation
    d_value = cohens_d(scores_a, scores_b)
    cohens_d_interpretation = ""
    if np.isinf(d_value):
        cohens_d_interpretation = "Infinite effect size (likely due to zero variance in scores)."
    elif abs(d_value) >= 0.8:
        cohens_d_interpretation = "Large effect size."
    elif abs(d_value) >= 0.5:
        cohens_d_interpretation = "Medium effect size."
    elif abs(d_value) >= 0.2:
        cohens_d_interpretation = "Small effect size."
    else:
        cohens_d_interpretation = "Negligible effect size."

    # McNemar's test calculation
    preds_a = all_predictions[name_a]
    preds_b = all_predictions[name_b]

    correct_a = (preds_a == y_test)
    correct_b = (preds_b == y_test)

    n_00 = ((correct_a == True) & (correct_b == True)).sum()
    n_01 = ((correct_a == True) & (correct_b == False)).sum()
    n_10 = ((correct_a == False) & (correct_b == True)).sum()
    n_11 = ((correct_a == False) & (correct_b == False)).sum()

    contingency_table = [[n_00, n_01],
                         [n_10, n_11]]

    # Handle cases where mcnemar might fail with very small N
    try:
        mcnemar_result = mcnemar(contingency_table, exact=False)
        mcnemar_statistic = mcnemar_result.statistic
        mcnemar_pvalue = mcnemar_result.pvalue
        mcnemar_conclusion = "Statistically significant difference (reject H0)." if mcnemar_pvalue < alpha else "No statistically significant difference (fail to reject H0)."
    except ValueError:
        mcnemar_statistic = 'N/A'
        mcnemar_pvalue = 'N/A'
        mcnemar_conclusion = "McNemar's test could not be performed due to insufficient data or conditions."

    combined_results.append({
        'Pair': f"{name_a} vs {name_b}",
        'McNemar_Statistic': mcnemar_statistic,
        'McNemar_PValue': mcnemar_pvalue,
        'McNemar_Conclusion': mcnemar_conclusion,
        'Cohens_d': d_value,
        'Cohens_d_Interpretation': cohens_d_interpretation
    })

# Generate Markdown report
markdown_report = "# Combined Statistical Analysis: McNemar's Test and Cohen's d\n\n"
markdown_report += "This section presents a combined analysis of classifier performance differences using McNemar's Test for statistical significance and Cohen's d for effect size. The F1-macro scores from 10-fold cross-validation on the training set were used for Cohen's d, while predictions on the test set were used for McNemar's Test.\n\n"

# Add a note about the SVM cross-validation scores, if still relevant based on the CV scores.
if np.std(scores_svm_unbal) < 0.01 or np.std(scores_svm_bal) < 0.01: # Small std indicates potential issue
    markdown_report += "**Important Note on SVM Cross-Validation Scores:** It was observed that the cross-validation F1-macro scores for SVM models (both unbalanced and balanced) are very consistent, resulting in extremely small standard deviations. This leads to exceptionally large Cohen's d values when compared against other classifiers, and potentially infinite if the standard deviation was exactly zero. This suggests a potential issue with the cross-validation setup or the SVM model's performance stability across training folds, making the Cohen's d interpretation for pairs involving SVM unreliable in terms of magnitude. The high GridSearchCV scores for SVM suggest good performance, but the CV scores variability is low.\n\n"

for result in combined_results:
    markdown_report += f"## Pair: {result['Pair']}\n"
    markdown_report += f"### McNemar's Test\n"
    markdown_report += f"- **Statistic (χ²):** {result['McNemar_Statistic']:.4f}\n"
    markdown_report += f"- **p-value:** {result['McNemar_PValue']:.4f}\n"
    markdown_report += f"- **Conclusion (α={alpha}):** {result['McNemar_Conclusion']}\n"
    markdown_report += f"### Cohen's d\n"
    markdown_report += f"- **Effect Size (d):** {result['Cohens_d']:.4f}\n"
    markdown_report += f"- **Interpretation:** {result['Cohens_d_Interpretation']}\n\n"

# Print the markdown report (this will be rendered by the notebook)
print(markdown_report)


In [ ]:
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
ax = plt.gca()

# Plot ROC curve for Logistic Regression
lr_disp = RocCurveDisplay.from_estimator(best_lr, X_test_scaled, y_test, name='Logistic Regression', alpha=0.8, lw=2, ax=ax, ls='-')

# Plot ROC curve for Random Forest
rf_disp = RocCurveDisplay.from_estimator(best_rf, X_test_scaled, y_test, name='Random Forest', alpha=0.8, lw=2, ax=ax, ls='--')

# Plot ROC curve for SVM
svm_disp = RocCurveDisplay.from_estimator(best_svm, X_test_scaled, y_test, name='SVM', alpha=0.8, lw=2, ax=ax, ls='-.')

# Add the diagonal dashed black line for a random classifier
plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='black', label='Random Classifier (AUC = 0.5)', alpha=0.8)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Classification Models')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('fig2_roc_curves.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Define class labels for better readability
class_names = ['Legitimate', 'Phishing'] # Assuming -1 maps to Legitimate, 1 maps to Phishing

# Create a figure with 1 row and 3 columns for the subplots
fig, axes = plt.subplots(1, 3, figsize=(21, 6)) # Increased width for 3 subplots

# List of models and their predictions
models = {
    'Logistic Regression': pred_lr,
    'Random Forest': pred_rf,
    'SVM': pred_svm
}

for i, (model_name, predictions) in enumerate(models.items()):
    cm = confusion_matrix(y_test, predictions)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=class_names, yticklabels=class_names)

    axes[i].set_title(f'{model_name} Confusion Matrix')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('fig3_confusion_matrices.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Get feature importances from the best Random Forest model
feature_importances = best_rf.feature_importances_

# Create a Series linking feature names to their importances
features_df = pd.Series(feature_importances, index=X.columns)

# Sort in descending order and take the top 15
top_15_features = features_df.nlargest(15)

# Create the horizontal bar chart
plt.figure(figsize=(10, 7)) # Adjust size for better readability of 15 features
sns.barplot(x=top_15_features.values, y=top_15_features.index, palette='viridis')

# Invert y-axis to have the most important feature at the top
plt.gca().invert_yaxis()

plt.xlabel('Feature Importance (Gini Impurity Decrease)')
plt.ylabel('Feature Name')
plt.title('Top 15 Feature Importances from Random Forest Classifier')
plt.tight_layout()
plt.savefig('fig4_feature_importance.pdf', dpi=300, bbox_inches='tight')
plt.show()

caption = "Figure 4: Top 15 feature importances as determined by the Random Forest classifier, based on Gini impurity decrease. Features are sorted in descending order of importance, with the most impactful feature at the top. This visualization helps identify the most significant characteristics distinguishing phishing websites from legitimate ones. The dataset size is n=11055."
print(caption)